# install

In [8]:
# Cài đặt hoặc nâng cấp vnstock
!pip install -U vnstock

In [9]:
from vnstock import Quote
quote = Quote(symbol='ACB', source='KBS')

from vnstock import Listing
# Khởi tạo đối tượng Listing với nguồn VCI
listing = Listing(source='VCI')
listing.symbols_by_group('VN30')


0     ACB
1     BID
2     BSR
3     CTG
4     FPT
5     GAS
6     GVR
7     HDB
8     HPG
9     LPB
10    MBB
11    MCH
12    MSN
13    MWG
14    SAB
15    SHB
16    SSB
17    SSI
18    STB
19    TCB
20    TCX
21    VCB
22    VHM
23    VIB
24    VIC
25    VJC
26    VNM
27    VPB
28    VPL
29    VRE

(vì đang sử dụng collab, API từ VCI bị hạn chế, nên chỉ có thể để kết quả ở dạng text)

Các mã bên dưới lựa chọn đều thuộc danh sách VN30 hiện tại (có thể gây ra survivor bias)

Có các mã được bổ sung: MCH, TCX
Các mã bị bỏ: VCK, PLX, TPB, trước đó nữa thì có DGC cũng bị loại

# Cell 1: Load dữ liệu

In [10]:
### Cell 1: Load dữ liệu (price + volume)
from vnstock import Quote
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

symbols = ["BID", "ACB", "VCB", "VNM", "MSN", "MWG", "HPG", "GAS", "SSI", "VRE"]

raw = {}
for sym in symbols:
    df = Quote(symbol=sym, source="KBS").history(start="2024-01-01", end="2025-12-31", interval="d")
    raw[sym] = df.set_index("time")[["close", "volume"]]

# start từ 2024 để có đủ 252 phiên lookback cho factor momentum tính trong 2025
price_df = pd.DataFrame({sym: d["close"] for sym, d in raw.items()}).sort_index()
volume_df = pd.DataFrame({sym: d["volume"] for sym, d in raw.items()}).sort_index()

price_df.head()

,BID,ACB,VCB,VNM,MSN,MWG,HPG,GAS,SSI,VRE
time,,,,,,,,,,
2024-01-02 07:00:00,34.75,14.80,55.04,57.89,68.4,40.88,18.55,64.82,22.57,22.31
2024-01-03 07:00:00,35.40,15.13,55.70,58.49,68.9,41.60,18.78,65.17,22.88,22.45
2024-01-04 07:00:00,35.28,15.32,56.63,58.49,68.1,41.60,18.75,65.77,23.34,22.60
2024-01-05 07:00:00,35.97,15.41,56.82,58.32,67.9,42.23,18.78,66.19,23.72,22.55
2024-01-08 07:00:00,37.50,15.34,57.22,57.81,66.6,41.60,18.82,65.85,23.68,22.89


# Cell 2: Check data

In [11]:
### Cell 2: Kiểm tra các phiên biến động giá > 7%
def find_extreme_moves(price: pd.DataFrame, threshold: float = 0.07) -> pd.DataFrame:
    """Tìm các phiên có |return| > threshold.

    Returns:
        DataFrame với columns [date, symbol, return] cho các phiên vượt ngưỡng.
    """
    ret = price.pct_change()
    mask = ret.abs() > threshold
    return (
        ret[mask]
        .stack()
        .rename("return")
        .reset_index()
        .rename(columns={"level_0": "date", "level_1": "symbol"})
        .sort_values("return", key=abs, ascending=False)
        .reset_index(drop=True)
    )

extreme_moves = find_extreme_moves(price_df)
print(f"Số phiên biến động > 7%: {len(extreme_moves)}")
extreme_moves

Số phiên biến động > 7%: 3


,time,symbol,return
0,2025-04-08 07:00:00,VCB,-0.070125
1,2025-04-03 07:00:00,HPG,-0.070089
2,2025-09-30 07:00:00,VRE,0.070072


Kết quả vẫn nằm trong khoảng chấp nhận được, sàn HOSE có khoảng dao động 7%, kết quả này hoàn toàn chấp nhận được

# Cell 3: Momentum factor và IC mới, sau khi loại đi cổ phiếu đặc biệt

In [12]:
### Cell 3b: Loại VRE khỏi dữ liệu, tính lại factor và IC
N_FWD = 5

def calc_ic(factor_df: pd.DataFrame, fwd_ret_df: pd.DataFrame) -> pd.Series:
    """Cross-sectional Spearman IC giữa factor và forward return, theo từng ngày."""
    ic = {}
    for date in factor_df.index:
        f, r = factor_df.loc[date], fwd_ret_df.loc[date]
        valid = f.notna() & r.notna()
        if valid.sum() >= 5:  # cần đủ số mã để correlation có ý nghĩa
            ic[date] = spearmanr(f[valid], r[valid])[0]
    return pd.Series(ic).dropna()

price_df = price_df.drop(columns="VRE")
volume_df = volume_df.drop(columns="VRE")

mom_12_1 = price_df.shift(21) / price_df.shift(252) - 1
fwd_ret = price_df.shift(-N_FWD) / price_df - 1

factors = {"Momentum_12_1": mom_12_1}

ic_series = {name: calc_ic(f, fwd_ret) for name, f in factors.items()}

ic_summary = pd.DataFrame({
    name: {
        "Mean IC": s.mean(),
        "Std IC": s.std(),
        "IC IR": s.mean() / s.std(),
        "t-stat": s.mean() / s.std() * np.sqrt(len(s)),
        "Hit Rate": (s > 0).mean(),
        "N Obs": len(s),
    }
    for name, s in ic_series.items()
}).T.round(3)

ic_summary

,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
Momentum_12_1,-0.004,0.395,-0.01,-0.152,0.525,242.0


Momentum cũ có mean IC là -0.045, lần này nếu loại đi cổ phiếu VRE, IC tụt mạnh về còn 0.004 và vẫn mang dấu âm

# Cell 4: Plot so sánh

In [13]:
### Cell 4: Khôi phục lại VRE (bổ sung lại vào price_df/volume_df sau khi đã drop ở Cell 3b)
vre = Quote(symbol="VRE", source="KBS").history(start="2024-01-01", end="2025-12-31", interval="d")
vre = vre.set_index("time")[["close", "volume"]]

price_df = price_df.join(vre["close"].rename("VRE"))
volume_df = volume_df.join(vre["volume"].rename("VRE"))

price_df = price_df.sort_index(axis=1)
volume_df = volume_df.sort_index(axis=1)

price_df.tail()

,ACB,BID,GAS,HPG,MSN,MWG,SSI,VCB,VNM,VRE
time,,,,,,,,,,
2025-12-25 07:00:00,20.69,38.36,69.2,23.43,76.3,84.68,30.50,56.68,59.41,31.15
2025-12-26 07:00:00,20.61,38.36,70.5,24.01,75.3,85.67,30.75,56.68,59.60,30.90
2025-12-29 07:00:00,20.69,38.36,75.1,23.83,75.5,85.76,30.50,56.68,60.18,32.06
2025-12-30 07:00:00,20.78,38.95,74.9,23.65,76.9,87.14,30.60,56.88,59.89,31.68
2025-12-31 07:00:00,20.69,38.46,72.4,23.56,77.0,87.04,30.25,57.08,59.31,32.50


# Cell 5-6: bổ sung hàm, dữ liệu chỉ báo


In [14]:
### Cell 5: Helper functions dùng chung cho các alpha (style Alpha101-ish)
def delta(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.diff(d)

def delay(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.shift(d)

def ts_sum(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).sum()

def ts_min(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).min()

def ts_max(df: pd.DataFrame, d: int) -> pd.DataFrame:
    return df.rolling(d).max()

def rank(df: pd.DataFrame) -> pd.DataFrame:
    """Cross-sectional percentile rank, theo từng ngày (0-1)."""
    return df.rank(axis=1, pct=True)

def calc_rsi(price: pd.DataFrame, window: int = 14) -> pd.DataFrame:
    """RSI Wilder's smoothing, vectorized theo cột."""
    delta_p = price.diff()
    gain = delta_p.clip(lower=0)
    loss = -delta_p.clip(upper=0)
    avg_gain = gain.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1 / window, min_periods=window, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - 100 / (1 + rs)

In [15]:
### Cell 6: Tính RSI và forward return (recompute sau khi thêm lại VRE)
rsi_df = calc_rsi(price_df, window=14)
fwd_ret = price_df.shift(-N_FWD) / price_df - 1

factors = {}
ic_series = {}

# Cell 7-11: Các chiến lược alpha

In [16]:
### Cell 7: Alpha C — trend-condition ternary
# Nếu xu hướng giá 100 ngày gần như đi ngang (<=5%) -> khoảng cách tới đáy 100 ngày
# Ngược lại (trend rõ) -> momentum ngắn hạn 3 ngày đảo dấu
trend_cond = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)) <= 0.05
alpha_c = pd.DataFrame(
    np.where(trend_cond, -1 * (price_df - ts_min(price_df, 100)), -1 * delta(price_df, 3)),
    index=price_df.index, columns=price_df.columns
)

factors["AlphaC_TrendCond"] = alpha_c
ic_series["AlphaC_TrendCond"] = calc_ic(alpha_c, fwd_ret)

- Alpha C (ý tưởng từ WorldQuant)

đi theo chế độ thị trường

càng gần đáy thì càng nên mua

nếu có trend => dùng mean reversion để tránh đu đỉnh

In [17]:
### Cell 8: Alpha D (pure, chưa regime) & Alpha D4 (regime switch — extreme override)
SLOPE_WINDOW = 5

rsi_slope = (rsi_df - rsi_df.shift(SLOPE_WINDOW)) / SLOPE_WINDOW
rsi_high_100 = rsi_df >= ts_max(rsi_df, 100)
rsi_low_100 = rsi_df <= ts_min(rsi_df, 100)

# Alpha D (pure): trend-following theo slope RSI, chưa có override cực trị
alpha_d_pure = pd.DataFrame(np.sign(rsi_slope), index=rsi_df.index, columns=rsi_df.columns)

# Alpha D4 (regime switch): override reversal khi RSI chạm cực trị 100 ngày (đỉnh->bán, đáy->mua),
# ngược lại fallback theo slope (Alpha D pure)
alpha_d4 = pd.DataFrame(
    np.select(
        [rsi_high_100.values, rsi_low_100.values],
        [-1, 1],
        default=alpha_d_pure.values,
    ),
    index=rsi_df.index, columns=rsi_df.columns,
)

factors["AlphaD_NoRegime"] = alpha_d_pure
factors["AlphaD4_RegimeSwitch"] = alpha_d4
ic_series["AlphaD_NoRegime"] = calc_ic(alpha_d_pure, fwd_ret)
ic_series["AlphaD4_RegimeSwitch"] = calc_ic(alpha_d4, fwd_ret)

An input array is constant; the correlation coefficient is not defined.


**Alpha D (No Regime)**
đi theo độ dốc của RSI (RSI slope)

RSI tăng → mua

RSI giảm → bán

RSI đi ngang → không vị thế

=> Đây là chiến lược trend-following theo momentum của RSI, chưa xét RSI đang ở vùng cực trị.

**Alpha D4 (Regime Switch)**

đi theo độ dốc RSI, nhưng có override khi RSI chạm cực trị 100 ngày

RSI ở đỉnh 100 ngày → bán (reversal, tránh đu đỉnh)

RSI ở đáy 100 ngày → mua (reversal, bắt đáy)

nếu không ở cực trị → fallback theo RSI slope

RSI tăng → mua

RSI giảm → bán

RSI đi ngang → không vị thế

→ Ý tưởng là trend-following bình thường, nhưng chuyển sang mean reversion khi RSI ở vùng cực trị, nhằm tránh mua quá cao hoặc bán quá thấp.

In [18]:
### Cell 9: Alpha D5 — regime blend liên tục + vùng cực trị nới lỏng theo percentile
trend_strength = (delta(ts_sum(price_df, 100) / 100, 100) / delay(price_df, 100)).abs()
trend_weight = np.tanh(trend_strength / 0.05)  # normalize quanh ngưỡng gốc 5%, nén về [0,1]

rsi_range_100 = ts_max(rsi_df, 100) - ts_min(rsi_df, 100)
rsi_pct_pos = (rsi_df - ts_min(rsi_df, 100)) / rsi_range_100
extreme_high = rsi_pct_pos >= 0.9
extreme_low = rsi_pct_pos <= 0.1

slope_signal_continuous = np.tanh(rsi_slope / 2)
blended_signal = trend_weight * slope_signal_continuous

alpha_d5 = pd.DataFrame(
    np.select(
        [extreme_high.values, extreme_low.values],
        [-1, 1],
        default=blended_signal.values,
    ),
    index=rsi_df.index, columns=rsi_df.columns,
)

factors["AlphaD5_SoftRegime"] = alpha_d5
ic_series["AlphaD5_SoftRegime"] = calc_ic(alpha_d5, fwd_ret)

**Alpha D5 (Soft Regime Blend)**

kết hợp trend-following + mean reversion một cách liên tục, thay vì chuyển regime cứng

trend mạnh → tăng trọng số cho RSI slope

trend yếu → giảm trọng số trend-following → tín hiệu yếu hơn

- dùng percentile của RSI trong vùng 100 ngày để xác định cực trị:

RSI nằm top 10% → bán (reversal, tránh đu đỉnh)

RSI nằm bottom 10% → mua (reversal, bắt đáy)

vùng giữa → đi theo độ dốc RSI

cường độ tín hiệu được làm mềm bằng tanh:

RSI slope tăng → tín hiệu mua mạnh dần

RSI slope giảm → tín hiệu bán mạnh dần

trend càng mạnh → tín hiệu slope càng được khuếch đại

→ Ý tưởng chính: trend càng mạnh thì càng tin vào momentum của RSI; khi RSI rơi vào vùng cực trị thì ưu tiên mean reversion để tránh đu đỉnh/bắt đáy quá muộn.

In [19]:
### Cell 10: Ensemble Alpha C + Alpha D4 (regime switch) và Alpha C + Alpha D5 (soft regime)
# Chuẩn hóa Alpha C về [-1,1] bằng cross-sectional rank để cùng thang đo với alpha D4/D5
alpha_c_scaled = 2 * rank(alpha_c) - 1

alpha_ensemble_c_d4 = (alpha_c_scaled + alpha_d4) / 2
alpha_ensemble_c_d5 = (alpha_c_scaled + alpha_d5) / 2

factors["Ensemble_C_D4"] = alpha_ensemble_c_d4
factors["Ensemble_C_D5"] = alpha_ensemble_c_d5
ic_series["Ensemble_C_D4"] = calc_ic(alpha_ensemble_c_d4, fwd_ret)
ic_series["Ensemble_C_D5"] = calc_ic(alpha_ensemble_c_d5, fwd_ret)

**Ensemble Alpha C + Alpha D4**

kết hợp Alpha C (mean reversion theo chế độ thị trường) với Alpha D4 (trend-following + reversal tại cực trị)

Alpha C được rank cross-sectional về [-1, 1] để cùng thang đo với Alpha D4

sau đó lấy trung bình 50/50 giữa hai alpha

→ Ý tưởng: kết hợp tín hiệu mean reversion của Alpha C với tín hiệu RSI, để hai nguồn tín hiệu bổ trợ nhau.

khi Alpha C và D4 cùng hướng → tín hiệu mạnh hơn

khi ngược hướng → hai tín hiệu triệt tiêu nhau, giảm độ mạnh của vị thế

**Ensemble Alpha C + Alpha D5**

kết hợp Alpha C với Alpha D5 (soft regime blend)

Alpha D5 không chuyển regime cứng mà pha trộn liên tục trend-following và mean reversion dựa trên độ mạnh của trend.

sau đó lấy trung bình 50/50:

Ensemble = 0.5 × Alpha C + 0.5 × Alpha D5

→ Ý tưởng: kết hợp một alpha thiên về mean reversion với một alpha có khả năng thích nghi theo trend, giúp tín hiệu linh hoạt hơn.

D4 → regime switch cứng: gặp cực trị RSI thì đảo chiều ngay.

D5 → regime switch mềm: điều chỉnh cường độ tín hiệu liên tục theo trend và percentile RSI.

Cell 11: IC các chiến lược

In [20]:
### Cell 11: Bảng so sánh IC — 6 chiến lược yêu cầu
def ic_stats(s: pd.Series) -> dict:
    return {
        "Mean IC": s.mean(),
        "Std IC": s.std(),
        "IC IR": s.mean() / s.std(),
        "t-stat": s.mean() / s.std() * np.sqrt(len(s)),
        "Hit Rate": (s > 0).mean(),
        "N Obs": len(s),
    }

compare_final = pd.DataFrame({
    name: ic_stats(s) for name, s in ic_series.items()
}).T.round(4)

compare_final

,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
AlphaC_TrendCond,0.0643,0.3629,0.1772,3.9262,0.5703,491.0
AlphaD_NoRegime,-0.0033,0.3476,-0.0095,-0.1905,0.4774,398.0
AlphaD4_RegimeSwitch,0.0124,0.3482,0.0355,0.7271,0.4976,420.0
AlphaD5_SoftRegime,0.0173,0.3225,0.0537,0.9226,0.5119,295.0
Ensemble_C_D4,0.0543,0.3233,0.1680,3.6622,0.5537,475.0
Ensemble_C_D5,0.0714,0.3431,0.2080,3.5726,0.5492,295.0


Các chiến lược alpha D (D, D4, D5) khi đứng độc lập đều không mang ý nghĩa thống kê rõ ràng

Khi kết hợp với chiến lược C, kết quả tăng lên đáng kể

ICIR của 2 chiến lược ensemsle đều tăng cao hơn Alpha C, và std được giảm đi một chút, tuy nhiên đánh đổi là hit rate giảm nhẹ

=> Việc chạy theo trend nhờ 1 chỉ báo momentum như RSI có tương quan âm với chiến lược mean-reversion (alpha C), kết hợp 2 chiến lược khả năng sẽ giúp giảm rủi ro tốt hơn

# Cell 12: Kiểm tra số quan sát

In [21]:
### Cell 12: Kiểm tra đồng nhất số quan sát (N Obs) giữa các signal
n_obs = compare_final["N Obs"]
if n_obs.nunique() > 1:
    print("⚠ Các signal KHÔNG cùng số quan sát:")
    print(n_obs)
    print(f"\nChênh lệch có thể do lookback khác nhau: Alpha C cần 252 phiên (momentum-style ts_sum/delta 100+100), "
          f"Alpha D/D4/D5 chỉ cần ~100-119 phiên (RSI 14 + rolling 100).")
else:
    print("✓ Tất cả signal có cùng số quan sát:", n_obs.iloc[0])

⚠ Các signal KHÔNG cùng số quan sát:
AlphaC_TrendCond        491.0
AlphaD_NoRegime         398.0
AlphaD4_RegimeSwitch    420.0
AlphaD5_SoftRegime      295.0
Ensemble_C_D4           475.0
Ensemble_C_D5           295.0
Name: N Obs, dtype: float64

Chênh lệch có thể do lookback khác nhau: Alpha C cần 252 phiên (momentum-style ts_sum/delta 100+100), Alpha D/D4/D5 chỉ cần ~100-119 phiên (RSI 14 + rolling 100).


Alpha C cần nhiều quan sát (gần như cả năm) để làm nóng

# Cell 13: So sánh trên cùng số quan sát

In [22]:
### Cell 13: So sánh lại trên cùng tập observation (common dates)
# Nếu N Obs lệch nhau, so IC trực tiếp là KHÔNG công bằng — signal có ít lookback hơn (RSI-based)
# được test trên giai đoạn thị trường khác với signal cần lookback dài (momentum-based),
# nên Mean IC/Sharpe của chúng không thể so trực tiếp nếu regime thị trường khác nhau giữa 2 giai đoạn.
common_idx = None
for s in ic_series.values():
    common_idx = s.index if common_idx is None else common_idx.intersection(s.index)

print(f"Common obs: {len(common_idx)} (riêng lẻ dao động {n_obs.min()}–{n_obs.max()})")

compare_common = pd.DataFrame({
    name: ic_stats(s.loc[common_idx]) for name, s in ic_series.items()
}).T.round(4)

compare_common

Common obs: 248 (riêng lẻ dao động 295.0–491.0)


,Mean IC,Std IC,IC IR,t-stat,Hit Rate,N Obs
AlphaC_TrendCond,0.0603,0.3656,0.1648,2.5960,0.5806,248.0
AlphaD_NoRegime,0.0183,0.3407,0.0538,0.8467,0.5000,248.0
AlphaD4_RegimeSwitch,0.0310,0.3416,0.0907,1.4276,0.5161,248.0
AlphaD5_SoftRegime,0.0122,0.3293,0.0371,0.5845,0.5000,248.0
Ensemble_C_D4,0.0716,0.3161,0.2266,3.5687,0.5766,248.0
Ensemble_C_D5,0.0725,0.3499,0.2072,3.2630,0.5484,248.0


Trên cùng 1 số lượng quan sát (248), 2 chiến lược ensemble có Mean IC cao hơn, std thấp hơn, t-stat chứng minh 2 chiến lược có ý nghĩa thống kê

Đặc biệt, chiến lược C+D4 mang lại hit rate 0.5766 ko kém quá nhiều so với 0.58 của alpha C

Điểm bất ngờ: Chiến lược regime cứng lại hiệu quả hơn theo hit rate

# Cell 14: Kiểm định multiple

In [24]:
### Cell 14: Multiple testing correction — Bonferroni & Benjamini-Hochberg (FDR)
# Lý do cần: đang so sánh 6 signal cùng lúc trên cùng bộ data -> p-value của từng signal riêng lẻ
# sẽ bị lạc quan (data snooping / multiple comparisons problem). Signal có t-stat cao "nhất trong 6"
# có thể chỉ do may mắn nếu không điều chỉnh ngưỡng ý nghĩa.
from scipy.stats import norm

n_tests = len(compare_common)
alpha = 0.05

# p-value 2 phía từ t-stat (xấp xỉ chuẩn vì N Obs đủ lớn)
p_values = pd.Series(
    {name: 2 * (1 - norm.cdf(abs(row["t-stat"]))) for name, row in compare_common.iterrows()}
)

# Bonferroni: ngưỡng chặt hơn theo số lượng test — kiểm soát Family-Wise Error Rate
bonferroni_threshold = alpha / n_tests
sig_bonferroni = p_values < bonferroni_threshold

# Benjamini-Hochberg (FDR): ít bảo thủ hơn Bonferroni, kiểm soát tỷ lệ false discovery kỳ vọng
sorted_p = p_values.sort_values()
bh_threshold = pd.Series(
    [(i + 1) / n_tests * alpha for i in range(n_tests)], index=sorted_p.index
)
sig_bh = sorted_p <= bh_threshold
# BH rule: signal có p-value nhỏ nhất thỏa điều kiện, và mọi signal có p nhỏ hơn nó cũng được coi là significant
if sig_bh.any():
    max_sig_rank = sig_bh[sig_bh].index  # các mã pass threshold tại đúng vị trí rank của nó
    cutoff_p = sorted_p[sig_bh].max() if sig_bh.any() else 0
    sig_bh_final = p_values <= cutoff_p
else:
    sig_bh_final = pd.Series(False, index=p_values.index)

correction_table = pd.DataFrame({
    "p-value": p_values,
    "Significant (raw, α=0.05)": p_values < alpha,
    f"Significant (Bonferroni, α={bonferroni_threshold:.4f})": sig_bonferroni,
    "Significant (BH/FDR)": sig_bh_final,
}).round(4)

correction_table.sort_values("p-value")

,p-value,"Significant (raw, α=0.05)","Significant (Bonferroni, α=0.0083)",Significant (BH/FDR)
Ensemble_C_D4,0.0004,True,True,True
Ensemble_C_D5,0.0011,True,True,True
AlphaC_TrendCond,0.0094,True,False,True
AlphaD4_RegimeSwitch,0.1534,False,False,False
AlphaD_NoRegime,0.3972,False,False,False
AlphaD5_SoftRegime,0.5589,False,False,False


Kết quả raw, chưa điều chỉnh: Chỉ có chiến lược alpha C và 2 chiến lược ensemble với alpha C mới có ý nghĩa

=> Kết luận lại: 1 chỉ báo thuần như RSI ko mang quá nhiều thông tin

Chiến lược alpha C ko đủ mạnh để vượt qua 1 kiểm định nghiêm ngặt như Bonferroni (dù vẫn vượt qua được FDR)  với 6 chiến lược được test cùng

2 chiến lược khả quan nhất là ensemble alpha C và RSI. Dù 2 chiến lược D4 và D5 không có ý nghĩa thống kê khi đứng đơn lẻ, nhưng khi được kết hợp với alpha C, lại làm cho chiến lược kết hợp có bằng chứng thống kê có ý nghĩa.

Tức là, RSI vẫn có thể cung cấp thêm thông tin bổ sung, thay vì tự thân là 1 alpha mạnh.

=> Đúng với thực tế đầu tư: ko chỉ dùng 1 chỉ báo để ra chiến lược